# Extract virus detections (multi-source)

Builds a unified raw table under `data/01_raw/virus/` from programmatic feeds aligned with [`specs/02-virus-dataset-selection.md`](../specs/02-virus-dataset-selection.md).

| Source key | Feed | Status in this notebook |
|------------|------|-------------------------|
| `ena_sequence` / `ena_read_run` | ENA Portal `sequence` + `read_run` + BioSample backfill | **Implemented** |
| `bvbrc` | BV-BRC Genome API | **Implemented** |
| `pathoplexus` | Pathoplexus LAPIS | **Implemented** |
| `empres_i` | FAO EMPRES-i BigQuery CSV | **Implemented** |
| `ecdc_atlas` | ECDC WNV weekly/monthly NUTS3 CSVs | **Implemented** |

- Endpoints: ENA Portal; BV-BRC; Pathoplexus LAPIS; FAO EMPRES-i BigQuery CSV; ECDC `wnv-weekly` / `wnv-monthly`
- License / terms: [EMBL-EBI](https://www.ebi.ac.uk/about/terms-of-use/); [BV-BRC](https://www.bv-brc.org/); [Pathoplexus](https://pathoplexus.org/); [FAO EMPRES-i](https://empres-i.apps.fao.org/); [ECDC](https://www.ecdc.europa.eu/) (no account required)
- Shared helpers: `vector_guard.virus_extract` (`VIRUS_RAW_COLUMNS`, keep filter, INSDC + outbreak dedupe)

This notebook stops at the raw extract (TSV + manifest + smoke check). Cleaning, geocoding, and VectorNet joins come later.

In [1]:
from __future__ import annotations

import base64
import hashlib
import html as html_module
import io
import json
import re
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from tqdm import tqdm

from vector_guard.data import (
    VIRUS_VECTOR_TAXON_MAP,
    default_virus_raw_dir,
    find_repo_root,
    load_virus_detections,
    resolve_virus_core_path,
)
from vector_guard.virus_extract import (
    INSDC_SOURCES,
    OUTBREAK_SOURCES,
    VIRUS_RAW_COLUMNS,
    apply_keep_filter,
    assign_record_ids,
    dedupe_virus_detections,
    empty_virus_frame,
    ensure_virus_columns,
    lineage_tax_ids,
    normalize_accession,
    pathogen_lookup,
    pick_primary_accession,
    resolve_coords,
    split_country_subnational,
)

REPO_ROOT = find_repo_root()
RAW_DIR = default_virus_raw_dir(REPO_ROOT)
TSV_NAME = "virus_detections.tsv"
TSV_PATH = RAW_DIR / TSV_NAME
MANIFEST_PATH = RAW_DIR / "manifest.json"

ENA_SEARCH_URL = "https://www.ebi.ac.uk/ena/portal/api/search"
ENA_CITATION = (
    "European Nucleotide Archive (ENA), EMBL-EBI. Sequence / read-run / sample "
    "metadata retrieved via the ENA Portal API (INSDC: ENA + GenBank + DDBJ). "
    "https://www.ebi.ac.uk/ena/portal/api/"
)
BVBRC_GENOME_URL = "https://www.bv-brc.org/api/genome/"
BVBRC_CITATION = (
    "Bacterial and Viral Bioinformatics Resource Center (BV-BRC). Viral genome "
    "metadata retrieved via the Genome REST API. "
    "https://www.bv-brc.org/api/genome/"
)
PATHOPLEXUS_BASE_URL = "https://lapis.pathoplexus.org"
PATHOPLEXUS_CITATION = (
    "Pathoplexus. Arbovirus sample metadata retrieved via organism LAPIS "
    "sample/details TSV endpoints. https://lapis.pathoplexus.org/"
)
# LAPIS organism slugs → canonical pathogen names in VIRUS_VECTOR_TAXON_MAP.
PATHOPLEXUS_ORGANISMS: tuple[dict[str, str], ...] = (
    {"slug": "west-nile", "pathogen": "West Nile virus"},
    {"slug": "dengue", "pathogen": "Dengue virus"},
    {"slug": "chikungunya", "pathogen": "Chikungunya virus"},
    {"slug": "zika", "pathogen": "Zika virus"},
    {"slug": "cchf", "pathogen": "Crimean-Congo hemorrhagic fever virus"},
    {"slug": "yellow-fever", "pathogen": "Yellow fever virus"},
)
EMPRES_BIGQUERY_URL = "https://api.data.apps.fao.org/api/v2/bigquery"
EMPRES_SQL_URL = (
    "https://data.apps.fao.org/catalog/dataset/3ff164cd-8b44-46d3-8f88-92b0361c7878/"
    "resource/137a69a0-ad5f-48c3-b927-3a61d2c9a2ce/download/"
    "animal-major-disease-parameterized-query.sql"
)
EMPRES_CITATION = (
    "FAO EMPRES-i+. Animal major-disease outbreak events retrieved via the FAO "
    "BigQuery CSV API using animal-major-disease-parameterized-query.sql. "
    "https://empres-i.apps.fao.org/"
)
EMPRES_START_DATE = "2000-01-01"
# EMPRES disease label → canonical pathogen name in VIRUS_VECTOR_TAXON_MAP.
EMPRES_DISEASES: tuple[dict[str, str], ...] = (
    {"disease": "West Nile Fever", "pathogen": "West Nile virus"},
    {"disease": "Bluetongue", "pathogen": "Bluetongue virus"},
    {"disease": "Rift Valley fever", "pathogen": "Rift Valley fever virus"},
    {"disease": "Schmallenberg", "pathogen": "Schmallenberg virus"},
    {"disease": "African horse sickness", "pathogen": "African horse sickness virus"},
    {"disease": "Japanese Encephalitis", "pathogen": "Japanese encephalitis virus"},
)
ECDC_WEEKLY_URL = "https://wnv-weekly.ecdc.europa.eu/"
ECDC_MONTHLY_URL = "https://wnv-monthly.ecdc.europa.eu/"
ECDC_CITATION = (
    "European Centre for Disease Prevention and Control (ECDC). West Nile virus "
    "infection seasonal surveillance: weekly and monthly NUTS3/GAUL reports. "
    "https://wnv-weekly.ecdc.europa.eu/ ; https://wnv-monthly.ecdc.europa.eu/"
)
ECDC_PATHOGEN = "West Nile virus"
DOWNLOAD_TIMEOUT_S = 600
CHUNK_SIZE = 1024 * 256
USER_AGENT = "vector-guard/0.1"
MAX_RETRIES = 5
RETRY_BACKOFF_S = 2.0
SAMPLE_BATCH_SIZE = 1000
BVBRC_PAGE_SIZE = 25000

BVBRC_SELECT_FIELDS = [
    "genome_id",
    "genome_name",
    "taxon_id",
    "genbank_accessions",
    "biosample_accession",
    "bioproject_accession",
    "isolation_country",
    "geographic_location",
    "state_province",
    "latitude",
    "longitude",
    "collection_date",
    "collection_year",
    "host_name",
    "isolation_source",
]

TARGETS = VIRUS_VECTOR_TAXON_MAP

ENA_SEQUENCE_FIELDS = [
    "accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "collection_date",
    "host",
    "isolation_source",
    "sample_accession",
    "study_accession",
]
ENA_READ_RUN_FIELDS = [
    "run_accession",
    "sample_accession",
    "study_accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "lat",
    "lon",
    "collection_date",
    "host",
    "isolation_source",
]
ENA_SAMPLE_FIELDS = [
    "sample_accession",
    "study_accession",
    "scientific_name",
    "tax_id",
    "country",
    "location",
    "lat",
    "lon",
    "collection_date",
    "host",
    "isolation_source",
]

# Filled by each source section; merge/write consumes this dict later.
source_frames: dict[str, pd.DataFrame] = {}
source_stats: dict[str, dict] = {}

RAW_DIR, ENA_SEARCH_URL, BVBRC_GENOME_URL, PATHOPLEXUS_BASE_URL, EMPRES_BIGQUERY_URL, ECDC_WEEKLY_URL, len(TARGETS), len(VIRUS_RAW_COLUMNS)

(PosixPath('/workspaces/vector-guard/data/01_raw/virus'),
 'https://www.ebi.ac.uk/ena/portal/api/search',
 'https://www.bv-brc.org/api/genome/',
 'https://lapis.pathoplexus.org',
 'https://api.data.apps.fao.org/api/v2/bigquery',
 'https://wnv-weekly.ecdc.europa.eu/',
 26,
 19)

## Wipe local raw directory

Every full extract run replaces the previous local copy.

In [2]:
def wipe_raw_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)
    for child in path.iterdir():
        if child.name == ".gitkeep":
            continue
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    (path / ".gitkeep").touch(exist_ok=True)


wipe_raw_dir(RAW_DIR)
print(f"Wiped and recreated: {RAW_DIR}")

Wiped and recreated: /workspaces/vector-guard/data/01_raw/virus


## Shared HTTP helpers

Retrying ENA Portal GET/POST helpers used by the sequence, read-run, and sample steps.

In [3]:
def _http_body(
    request: Request,
    *,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
    retries: int = MAX_RETRIES,
) -> bytes:
    last_error: Exception | None = None
    for attempt in range(retries):
        try:
            with urlopen(request, timeout=timeout_s) as response:
                chunks: list[bytes] = []
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    chunks.append(chunk)
            return b"".join(chunks)
        except HTTPError as exc:
            last_error = exc
            # Do not retry client errors (e.g. invalid sample accessions → 400).
            if 400 <= int(exc.code) < 500:
                break
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
        except (URLError, TimeoutError) as exc:
            last_error = exc
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
    assert last_error is not None
    raise last_error


def ena_search_tsv(
    *,
    result: str,
    query: str,
    fields: list[str],
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """GET ENA Portal search results as a string-typed DataFrame."""
    params = {
        "result": result,
        "query": query,
        "fields": ",".join(fields),
        "format": "tsv",
        "limit": "0",
    }
    url = f"{ENA_SEARCH_URL}?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    if not text.strip():
        return pd.DataFrame(columns=fields)
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


import re

_SAMPLE_ACCESSION_RE = re.compile(
    r"^(?:SAMN\d+|SAMEA\d+|SAMD\d+|ERS\d+|SRS\d+|DRS\d+)$",
    re.IGNORECASE,
)


def _post_sample_batch(
    batch: list[str],
    *,
    fields: list[str],
    timeout_s: int,
) -> pd.DataFrame:
    """POST one BioSample accession batch; binary-split on HTTP 400 invalid IDs."""
    if not batch:
        return pd.DataFrame(columns=fields)
    data = urlencode(
        {
            "result": "sample",
            "includeAccessions": ",".join(batch),
            "fields": ",".join(fields),
            "format": "tsv",
            "limit": "0",
        }
    ).encode("utf-8")
    request = Request(
        ENA_SEARCH_URL,
        data=data,
        headers={
            "User-Agent": USER_AGENT,
            "Content-Type": "application/x-www-form-urlencoded",
        },
    )
    try:
        text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    except HTTPError as exc:
        if exc.code != 400 or len(batch) == 1:
            if exc.code == 400 and len(batch) == 1:
                return pd.DataFrame(columns=fields)
            raise
        mid = len(batch) // 2
        left = _post_sample_batch(batch[:mid], fields=fields, timeout_s=timeout_s)
        right = _post_sample_batch(batch[mid:], fields=fields, timeout_s=timeout_s)
        parts = [p for p in (left, right) if not p.empty]
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=fields)

    if not text.strip():
        return pd.DataFrame(columns=fields)
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


def ena_sample_by_accessions(
    accessions: list[str],
    *,
    fields: list[str] = ENA_SAMPLE_FIELDS,
    batch_size: int = SAMPLE_BATCH_SIZE,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """POST ENA Portal ``includeAccessions`` lookups for BioSamples."""
    unique = sorted(
        {
            a.strip()
            for a in accessions
            if str(a).strip() and _SAMPLE_ACCESSION_RE.match(str(a).strip())
        }
    )
    if not unique:
        return pd.DataFrame(columns=fields)

    frames: list[pd.DataFrame] = []
    for start in tqdm(range(0, len(unique), batch_size), desc="ENA samples", leave=False):
        batch = unique[start : start + batch_size]
        part = _post_sample_batch(batch, fields=fields, timeout_s=timeout_s)
        if not part.empty:
            frames.append(part)

    if not frames:
        return pd.DataFrame(columns=fields)
    out = pd.concat(frames, ignore_index=True)
    if "sample_accession" in out.columns:
        out = out.drop_duplicates(subset=["sample_accession"], keep="first")
    return out


print("ENA helpers ready")

ENA helpers ready


## ENA — `sequence` + `read_run` + sample backfill

For each pathogen in `VIRUS_VECTOR_TAXON_MAP`:

1. Fetch assembled sequences (`result=sequence`) and raw runs (`result=read_run`) via `tax_tree(<tax_id>)`.
2. Collect BioSample accessions and backfill missing place / time / host fields (`coord_source=sample_backfill` when coords come only from the sample).
3. Normalize into `VIRUS_RAW_COLUMNS` and apply the place-or-time keep filter.

Output: `ena_df` registered under `source_frames["ena"]`.

In [4]:
def _blank(series: pd.Series | None, index: pd.Index) -> pd.Series:
    if series is None:
        return pd.Series("", index=index, dtype=str)
    return series.fillna("").astype(str)


def _fill_blank(base: pd.Series, donor: pd.Series) -> pd.Series:
    base_clean = base.fillna("").astype(str).str.strip()
    donor_clean = donor.fillna("").astype(str).str.strip()
    return base_clean.where(base_clean != "", donor_clean)


def backfill_from_samples(df: pd.DataFrame, samples: pd.DataFrame) -> pd.DataFrame:
    """Fill missing ENA row fields from BioSample metadata keyed by sample_accession."""
    out = df.copy().reset_index(drop=True)
    if out.empty or samples.empty or "sample_accession" not in out.columns:
        return out

    sample_map = samples.drop_duplicates(subset=["sample_accession"], keep="first").set_index(
        "sample_accession"
    )
    keyed = out["sample_accession"].fillna("").astype(str).str.strip()
    matched = keyed.isin(sample_map.index)
    if not matched.any():
        return out

    # Align BioSample rows to the detection frame by position (same length/order).
    sample_rows = sample_map.reindex(keyed.to_numpy())
    sample_rows.index = out.index

    for column in (
        "country",
        "collection_date",
        "host",
        "isolation_source",
        "study_accession",
        "location",
    ):
        if column not in out.columns:
            out[column] = ""
        donor = _blank(sample_rows.get(column), out.index)
        out[column] = _fill_blank(_blank(out[column], out.index), donor)

    if "lat" not in out.columns:
        out["lat"] = ""
    if "lon" not in out.columns:
        out["lon"] = ""
    if "coord_source" not in out.columns:
        out["coord_source"] = "none"

    sample_lat = _blank(sample_rows.get("lat"), out.index)
    sample_lon = _blank(sample_rows.get("lon"), out.index)
    sample_location = _blank(sample_rows.get("location"), out.index)

    for idx in out.index[matched.to_numpy()]:
        current_lat = str(out.at[idx, "lat"]).strip()
        current_lon = str(out.at[idx, "lon"]).strip()
        if current_lat and current_lon:
            continue
        lat, lon, coord_source = resolve_coords(
            lat=sample_lat.at[idx],
            lon=sample_lon.at[idx],
            location=sample_location.at[idx],
            coord_source="sample_backfill",
        )
        if lat and lon:
            out.at[idx, "lat"] = lat
            out.at[idx, "lon"] = lon
            out.at[idx, "coord_source"] = coord_source

    return out

def normalize_ena_rows(
    df: pd.DataFrame,
    *,
    source: str,
    accession_column: str,
) -> pd.DataFrame:
    """Map an ENA flatfile frame into ``VIRUS_RAW_COLUMNS``."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        country_raw = str(raw.get("country", "") or "").strip()
        country_field, subnational = split_country_subnational(country_raw)

        existing_lat = str(raw.get("lat", "") or "").strip()
        existing_lon = str(raw.get("lon", "") or "").strip()
        existing_source = str(raw.get("coord_source", "") or "").strip()
        location = str(raw.get("location", "") or "").strip()

        if existing_lat and existing_lon and existing_source in {"exact_gps", "sample_backfill"}:
            lat, lon, coord_source = existing_lat, existing_lon, existing_source
        else:
            lat, lon, coord_source = resolve_coords(
                lat=existing_lat,
                lon=existing_lon,
                location=location,
                coord_source="exact_gps",
            )

        accession_value = normalize_accession(raw.get(accession_column, ""))
        sample_accession = str(raw.get("sample_accession", "") or "").strip()
        native = accession_value or sample_accession or str(raw.get("run_accession", "") or "").strip()

        rows.append(
            {
                "record_id": f"{source}:{native}" if native else source,
                "source": source,
                "accession": accession_value,
                "sample_accession": sample_accession,
                "study_accession": str(raw.get("study_accession", "") or "").strip(),
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": str(raw.get("scientific_name", "") or "").strip(),
                "tax_id": str(raw.get("tax_id", "") or "").strip(),
                "country": country_field,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": str(raw.get("collection_date", "") or "").strip(),
                "host": str(raw.get("host", "") or "").strip(),
                "isolation_source": str(raw.get("isolation_source", "") or "").strip(),
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("ENA normalize / backfill helpers ready")

ENA normalize / backfill helpers ready


In [5]:
sequence_frames: list[pd.DataFrame] = []
read_run_frames: list[pd.DataFrame] = []
ena_per_pathogen: dict[str, dict] = {}

for target in tqdm(TARGETS, desc="ENA pathogens"):
    pathogen = target["pathogen"]
    vector_group = target["vector_group"]
    seq_parts: list[pd.DataFrame] = []
    run_parts: list[pd.DataFrame] = []
    raw_sequence = 0
    raw_read_run = 0

    for tax_id in target["tax_ids"]:
        query = f"tax_tree({tax_id})"

        seq = ena_search_tsv(
            result="sequence", query=query, fields=ENA_SEQUENCE_FIELDS
        )
        if not seq.empty:
            seq = seq.copy()
            seq["query_tax_id"] = str(tax_id)
            seq["pathogen_name"] = pathogen
            seq["vector_group"] = vector_group
            seq_parts.append(seq)
            raw_sequence += len(seq)

        runs = ena_search_tsv(
            result="read_run", query=query, fields=ENA_READ_RUN_FIELDS
        )
        if not runs.empty:
            runs = runs.copy()
            runs["query_tax_id"] = str(tax_id)
            runs["pathogen_name"] = pathogen
            runs["vector_group"] = vector_group
            run_parts.append(runs)
            raw_read_run += len(runs)

    if seq_parts:
        sequence_frames.append(pd.concat(seq_parts, ignore_index=True))
    if run_parts:
        read_run_frames.append(pd.concat(run_parts, ignore_index=True))

    ena_per_pathogen[pathogen] = {
        "tax_ids": list(target["tax_ids"]),
        "raw_sequence": raw_sequence,
        "raw_read_run": raw_read_run,
    }

raw_sequence_df = (
    pd.concat(sequence_frames, ignore_index=True)
    if sequence_frames
    else pd.DataFrame(columns=[*ENA_SEQUENCE_FIELDS, "query_tax_id", "pathogen_name", "vector_group"])
)
raw_read_run_df = (
    pd.concat(read_run_frames, ignore_index=True)
    if read_run_frames
    else pd.DataFrame(columns=[*ENA_READ_RUN_FIELDS, "query_tax_id", "pathogen_name", "vector_group"])
)

# Drop exact duplicate query hits within each result type before sample backfill.
if not raw_sequence_df.empty:
    raw_sequence_df = raw_sequence_df.drop_duplicates(
        subset=["accession", "pathogen_name"], keep="first"
    ).reset_index(drop=True)
if not raw_read_run_df.empty:
    raw_read_run_df = raw_read_run_df.drop_duplicates(
        subset=["run_accession", "pathogen_name"], keep="first"
    ).reset_index(drop=True)

print(f"Raw ENA sequence rows: {len(raw_sequence_df):,}")
print(f"Raw ENA read_run rows: {len(raw_read_run_df):,}")

ENA pathogens:   0%|          | 0/26 [00:00<?, ?it/s]

ENA pathogens: 100%|██████████| 26/26 [00:38<00:00,  1.47s/it]

Raw ENA sequence rows: 144,032
Raw ENA read_run rows: 20,671


In [6]:
sample_ids = sorted(
    {
        *(raw_sequence_df.get("sample_accession", pd.Series(dtype=str)).fillna("").astype(str).str.strip()),
        *(raw_read_run_df.get("sample_accession", pd.Series(dtype=str)).fillna("").astype(str).str.strip()),
    }
    - {""}
)
print(f"Distinct sample_accession values: {len(sample_ids):,}")

sample_df = ena_sample_by_accessions(sample_ids)
print(f"BioSample rows returned: {len(sample_df):,}")

# Seed lat/lon/coord_source on sequence rows from the location string first.
if not raw_sequence_df.empty:
    seq_coords = [
        resolve_coords(location=loc, coord_source="exact_gps")
        for loc in raw_sequence_df.get("location", pd.Series("", index=raw_sequence_df.index))
    ]
    raw_sequence_df = raw_sequence_df.copy()
    raw_sequence_df["lat"] = [c[0] for c in seq_coords]
    raw_sequence_df["lon"] = [c[1] for c in seq_coords]
    raw_sequence_df["coord_source"] = [c[2] for c in seq_coords]

if not raw_read_run_df.empty:
    raw_read_run_df = raw_read_run_df.copy()
    run_coords = [
        resolve_coords(
            lat=lat,
            lon=lon,
            location=loc,
            coord_source="exact_gps",
        )
        for lat, lon, loc in zip(
            raw_read_run_df.get("lat", pd.Series("", index=raw_read_run_df.index)),
            raw_read_run_df.get("lon", pd.Series("", index=raw_read_run_df.index)),
            raw_read_run_df.get("location", pd.Series("", index=raw_read_run_df.index)),
            strict=True,
        )
    ]
    raw_read_run_df["lat"] = [c[0] for c in run_coords]
    raw_read_run_df["lon"] = [c[1] for c in run_coords]
    raw_read_run_df["coord_source"] = [c[2] for c in run_coords]

sequence_bf = backfill_from_samples(raw_sequence_df, sample_df)
read_run_bf = backfill_from_samples(raw_read_run_df, sample_df)

ena_sequence_df = apply_keep_filter(
    normalize_ena_rows(sequence_bf, source="ena_sequence", accession_column="accession")
)
ena_read_run_df = apply_keep_filter(
    normalize_ena_rows(read_run_bf, source="ena_read_run", accession_column="run_accession")
)

# Keep run accessions in record_id only — blank the sequence accession field so later
# INSDC merges group on true assembled accessions / sample_accession.
if not ena_read_run_df.empty:
    ena_read_run_df = ena_read_run_df.copy()
    ena_read_run_df["accession"] = ""

ena_df = ensure_virus_columns(
    pd.concat([ena_sequence_df, ena_read_run_df], ignore_index=True)
    if len(ena_sequence_df) or len(ena_read_run_df)
    else empty_virus_frame()
)
ena_df = assign_record_ids(ena_df)

for pathogen, stats in ena_per_pathogen.items():
    stats["kept_sequence"] = int((ena_sequence_df["pathogen_name"] == pathogen).sum()) if len(ena_sequence_df) else 0
    stats["kept_read_run"] = int((ena_read_run_df["pathogen_name"] == pathogen).sum()) if len(ena_read_run_df) else 0
    stats["kept_total"] = stats["kept_sequence"] + stats["kept_read_run"]

source_frames["ena"] = ena_df
source_stats["ena"] = {
    "sources": ["ena_sequence", "ena_read_run"],
    "citation": ENA_CITATION,
    "endpoint": ENA_SEARCH_URL,
    "raw_sequence": int(len(raw_sequence_df)),
    "raw_read_run": int(len(raw_read_run_df)),
    "biosamples_requested": len(sample_ids),
    "biosamples_returned": int(len(sample_df)),
    "kept_sequence": int(len(ena_sequence_df)),
    "kept_read_run": int(len(ena_read_run_df)),
    "kept_total": int(len(ena_df)),
    "coord_source_counts": ena_df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_pathogen": ena_per_pathogen,
}

print(f"ENA kept sequence rows: {len(ena_sequence_df):,}")
print(f"ENA kept read_run rows:  {len(ena_read_run_df):,}")
print(f"ENA combined rows:       {len(ena_df):,}")
print("coord_source counts:")
print(ena_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(ena_per_pathogen).T

Distinct sample_accession values: 16,125


BioSample rows returned: 15,897
ENA kept sequence rows: 118,407
ENA kept read_run rows:  17,111
ENA combined rows:       135,518
coord_source counts:
coord_source
none               125138
exact_gps            8189
sample_backfill      2191


,tax_ids,raw_sequence,raw_read_run,kept_sequence,kept_read_run,kept_total
West Nile virus,[3048448],25374,4132,8519,4095,12614
Usutu virus,[3048443],1995,418,1905,403,2308
Dengue virus,[3052464],60230,8376,57254,5225,62479
Chikungunya virus,[3426298],11200,2122,10052,2101,12153
Zika virus,[3048459],3378,3963,2245,3785,6030
Sindbis virus,[3426318],820,525,515,464,979
Rift Valley fever virus,[3052676],1912,140,1760,140,1900
Batai orthobunyavirus,[3052371],126,0,120,0,120
Tahyna / California encephalitis orthobunyavirus,"[3052445, 35305, 178787]",139,1,130,0,130
Inkoo orthobunyavirus,[45269],31,0,25,0,25


## BV-BRC — Genome REST API

1. Query `https://www.bv-brc.org/api/genome/` per pathogen with `in(taxon_lineage_ids,(...))` over ICTV + legacy tax IDs (`lineage_tax_ids`).
2. Page with `limit(25000, offset)` and `Accept: text/tsv` (`Content-Range` total).
3. Map GenBank / BioSample / BioProject / place / time / host fields; prefer an accession already seen in ENA via `pick_primary_accession`.
4. Tag `source=bvbrc`, apply the place/time keep filter, register `source_frames["bvbrc"]`.

Output: `bvbrc_df`.

In [7]:
_CONTENT_RANGE_RE = re.compile(r"items\s+(\d+)-(\d+)/(\d+|\*)", re.IGNORECASE)


def _http_body_and_headers(
    request: Request,
    *,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
    retries: int = MAX_RETRIES,
) -> tuple[bytes, dict[str, str]]:
    last_error: Exception | None = None
    for attempt in range(retries):
        try:
            with urlopen(request, timeout=timeout_s) as response:
                headers = {k: v for k, v in response.headers.items()}
                chunks: list[bytes] = []
                while True:
                    chunk = response.read(CHUNK_SIZE)
                    if not chunk:
                        break
                    chunks.append(chunk)
            return b"".join(chunks), headers
        except HTTPError as exc:
            last_error = exc
            if 400 <= int(exc.code) < 500:
                break
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
        except (URLError, TimeoutError) as exc:
            last_error = exc
            if attempt + 1 >= retries:
                break
            time.sleep(RETRY_BACKOFF_S * (2**attempt))
    assert last_error is not None
    raise last_error


def _parse_content_range_total(headers: dict[str, str]) -> int | None:
    raw = headers.get("Content-Range") or headers.get("content-range") or ""
    match = _CONTENT_RANGE_RE.search(raw)
    if not match:
        return None
    total = match.group(3)
    if total == "*":
        return None
    return int(total)


def bvbrc_genome_page(
    *,
    lineage_ids: list[int] | tuple[int, ...],
    fields: list[str] = BVBRC_SELECT_FIELDS,
    limit: int = BVBRC_PAGE_SIZE,
    offset: int = 0,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> tuple[pd.DataFrame, int | None]:
    """Fetch one BV-BRC genome TSV page; return ``(frame, total_or_None)``."""
    ids = ",".join(str(tax_id) for tax_id in lineage_ids)
    select = ",".join(fields)
    url = (
        f"{BVBRC_GENOME_URL}"
        f"?limit({int(limit)},{int(offset)})"
        f"&in(taxon_lineage_ids,({ids}))"
        f"&select({select})"
    )
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/tsv",
        },
    )
    body, headers = _http_body_and_headers(request, timeout_s=timeout_s)
    text = body.decode("utf-8", errors="replace")
    total = _parse_content_range_total(headers)
    if not text.strip():
        return pd.DataFrame(columns=fields), total
    frame = pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)
    return frame, total


def fetch_bvbrc_genomes_for_target(
    target: dict,
    *,
    page_size: int = BVBRC_PAGE_SIZE,
) -> pd.DataFrame:
    """Page through all BV-BRC genomes for one pathogen lineage union."""
    lineage_ids = lineage_tax_ids(target)
    frames: list[pd.DataFrame] = []
    offset = 0
    total: int | None = None

    while True:
        page, page_total = bvbrc_genome_page(
            lineage_ids=lineage_ids,
            limit=page_size,
            offset=offset,
        )
        if page_total is not None:
            total = page_total
        if page.empty:
            break
        page = page.copy()
        page["query_tax_id"] = str(target["tax_ids"][0])
        page["pathogen_name"] = target["pathogen"]
        page["vector_group"] = target["vector_group"]
        frames.append(page)
        offset += len(page)
        if total is not None and offset >= total:
            break
        if len(page) < page_size:
            break

    if not frames:
        return pd.DataFrame(
            columns=[*BVBRC_SELECT_FIELDS, "query_tax_id", "pathogen_name", "vector_group"]
        )
    return pd.concat(frames, ignore_index=True)


def normalize_bvbrc_rows(
    df: pd.DataFrame,
    *,
    known_accessions: set[str] | None = None,
) -> pd.DataFrame:
    """Map BV-BRC genome rows into ``VIRUS_RAW_COLUMNS`` (``source=bvbrc``)."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        genome_id = str(raw.get("genome_id", "") or "").strip()
        accession = pick_primary_accession(
            raw.get("genbank_accessions", ""),
            known_accessions=known_accessions,
        )
        sample_accession = str(raw.get("biosample_accession", "") or "").strip()
        study_accession = str(raw.get("bioproject_accession", "") or "").strip()

        geographic_location = str(raw.get("geographic_location", "") or "").strip()
        isolation_country = str(raw.get("isolation_country", "") or "").strip()
        state_province = str(raw.get("state_province", "") or "").strip()

        if geographic_location:
            country_field, subnational = split_country_subnational(geographic_location)
        elif isolation_country:
            country_field, subnational = split_country_subnational(isolation_country)
        else:
            country_field, subnational = "", ""

        if not country_field and isolation_country:
            country_field = isolation_country
        if not subnational and state_province:
            subnational = state_province

        lat, lon, coord_source = resolve_coords(
            lat=raw.get("latitude", ""),
            lon=raw.get("longitude", ""),
            coord_source="exact_gps",
        )

        collection_date = str(raw.get("collection_date", "") or "").strip()
        if not collection_date:
            collection_date = str(raw.get("collection_year", "") or "").strip()

        native = accession or genome_id or sample_accession
        rows.append(
            {
                "record_id": f"bvbrc:{native}" if native else "bvbrc",
                "source": "bvbrc",
                "accession": accession,
                "sample_accession": sample_accession,
                "study_accession": study_accession,
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": str(raw.get("genome_name", "") or "").strip(),
                "tax_id": str(raw.get("taxon_id", "") or "").strip(),
                "country": country_field,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": collection_date,
                "host": str(raw.get("host_name", "") or "").strip(),
                "isolation_source": str(raw.get("isolation_source", "") or "").strip(),
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("BV-BRC helpers ready")

BV-BRC helpers ready


In [8]:
known_ena_accessions: set[str] = set()
if "ena" in source_frames and not source_frames["ena"].empty:
    known_ena_accessions = {
        acc
        for acc in (
            normalize_accession(value) for value in source_frames["ena"]["accession"]
        )
        if acc
    }
print(f"Known ENA accessions for BV-BRC primary pick: {len(known_ena_accessions):,}")

bvbrc_raw_frames: list[pd.DataFrame] = []
bvbrc_per_pathogen: dict[str, dict] = {}

for target in tqdm(TARGETS, desc="BV-BRC pathogens"):
    pathogen = target["pathogen"]
    raw = fetch_bvbrc_genomes_for_target(target)
    if not raw.empty:
        raw = raw.drop_duplicates(subset=["genome_id", "pathogen_name"], keep="first")
        bvbrc_raw_frames.append(raw)
    bvbrc_per_pathogen[pathogen] = {
        "tax_ids": list(target["tax_ids"]),
        "legacy_tax_ids": list(target.get("legacy_tax_ids", ())),
        "lineage_tax_ids": list(lineage_tax_ids(target)),
        "raw": int(len(raw)),
    }

raw_bvbrc_df = (
    pd.concat(bvbrc_raw_frames, ignore_index=True)
    if bvbrc_raw_frames
    else pd.DataFrame(
        columns=[*BVBRC_SELECT_FIELDS, "query_tax_id", "pathogen_name", "vector_group"]
    )
)

bvbrc_df = apply_keep_filter(
    normalize_bvbrc_rows(raw_bvbrc_df, known_accessions=known_ena_accessions)
)
bvbrc_df = assign_record_ids(bvbrc_df)

for pathogen, stats in bvbrc_per_pathogen.items():
    stats["kept"] = (
        int((bvbrc_df["pathogen_name"] == pathogen).sum()) if len(bvbrc_df) else 0
    )

source_frames["bvbrc"] = bvbrc_df
source_stats["bvbrc"] = {
    "sources": ["bvbrc"],
    "citation": BVBRC_CITATION,
    "endpoint": BVBRC_GENOME_URL,
    "raw": int(len(raw_bvbrc_df)),
    "kept": int(len(bvbrc_df)),
    "known_ena_accessions": len(known_ena_accessions),
    "matched_ena_accession": int(
        bvbrc_df["accession"].isin(known_ena_accessions).sum()
    )
    if len(bvbrc_df) and known_ena_accessions
    else 0,
    "coord_source_counts": bvbrc_df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_pathogen": bvbrc_per_pathogen,
}

print(f"BV-BRC raw genome rows: {len(raw_bvbrc_df):,}")
print(f"BV-BRC kept rows:       {len(bvbrc_df):,}")
print("coord_source counts:")
print(bvbrc_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(
    {
        pathogen: {
            "raw": stats["raw"],
            "kept": stats["kept"],
        }
        for pathogen, stats in bvbrc_per_pathogen.items()
    }
).T

Known ENA accessions for BV-BRC primary pick: 118,404


BV-BRC pathogens: 100%|██████████| 26/26 [00:34<00:00,  1.33s/it]


BV-BRC raw genome rows: 152,125
BV-BRC kept rows:       125,338
coord_source counts:
coord_source
none    125338


,raw,kept
West Nile virus,25447,8516
Usutu virus,2047,1951
Dengue virus,62302,59126
Chikungunya virus,12184,11029
Zika virus,3428,2279
Sindbis virus,825,518
Rift Valley fever virus,1950,1792
Batai orthobunyavirus,181,175
Tahyna / California encephalitis orthobunyavirus,467,458
Inkoo orthobunyavirus,31,25


## Pathoplexus — LAPIS sample details

1. Fetch `https://lapis.pathoplexus.org/{slug}/sample/details?dataFormat=tsv` for `west-nile`, `dengue`, `chikungunya`, `zika`, `cchf`, `yellow-fever`.
2. Drop revocations / non-latest versions; map INSDC / BioSample / geo / date / host fields (CCHF uses segmented `insdcAccessionBase_{L,M,S}`).
3. Tag `source=pathoplexus`, apply the place/time keep filter, register `source_frames["pathoplexus"]`.

Output: `pathoplexus_df`.

In [9]:
def pathoplexus_sample_details(
    slug: str,
    *,
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """GET Pathoplexus LAPIS ``sample/details`` TSV for one organism slug."""
    url = f"{PATHOPLEXUS_BASE_URL}/{slug}/sample/details?dataFormat=tsv"
    request = Request(url, headers={"User-Agent": USER_AGENT, "Accept": "text/tab-separated-values,text/tsv,*/*"})
    text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    if not text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(text), sep="\t", dtype=str, keep_default_na=False)


def _pathoplexus_is_true(value: object) -> bool:
    return str(value or "").strip().lower() in {"true", "1", "yes"}


def _pathoplexus_insdc_accession(
    raw: pd.Series,
    *,
    known_accessions: set[str] | None = None,
) -> str:
    """Prefer plain ``insdcAccessionBase``; else CCHF L/M/S segment bases."""
    base = str(raw.get("insdcAccessionBase", "") or "").strip()
    if base:
        return pick_primary_accession(base, known_accessions=known_accessions)

    segments: list[str] = []
    for key in ("insdcAccessionBase_L", "insdcAccessionBase_M", "insdcAccessionBase_S"):
        part = str(raw.get(key, "") or "").strip()
        if part:
            segments.append(part)
    if not segments:
        return ""
    return pick_primary_accession(",".join(segments), known_accessions=known_accessions)


def _pathoplexus_isolation_source(raw: pd.Series) -> str:
    for key in (
        "anatomicalMaterial",
        "anatomicalPart",
        "bodyProduct",
        "sampleType",
        "environmentalMaterial",
    ):
        value = str(raw.get(key, "") or "").strip()
        if value:
            return value
    return ""


def normalize_pathoplexus_rows(
    df: pd.DataFrame,
    *,
    known_accessions: set[str] | None = None,
) -> pd.DataFrame:
    """Map Pathoplexus LAPIS rows into ``VIRUS_RAW_COLUMNS`` (``source=pathoplexus``)."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        accession = _pathoplexus_insdc_accession(raw, known_accessions=known_accessions)
        sample_accession = str(raw.get("biosampleAccession", "") or "").strip()
        study_accession = str(raw.get("bioprojectAccession", "") or "").strip()

        country_token = str(raw.get("geoLocCountry", "") or "").strip()
        admin1 = str(raw.get("geoLocAdmin1", "") or "").strip()
        if country_token and admin1:
            country_field = f"{country_token}: {admin1}"
            subnational = admin1
        else:
            country_field = country_token
            subnational = admin1

        lat, lon, coord_source = resolve_coords(
            lat=raw.get("geoLocLatitude", ""),
            lon=raw.get("geoLocLongitude", ""),
            coord_source="exact_gps",
        )

        host = str(raw.get("hostNameScientific", "") or "").strip()
        if not host:
            host = str(raw.get("hostNameCommon", "") or "").strip()

        scientific_name = str(raw.get("ncbiVirusName", "") or "").strip()
        if not scientific_name:
            scientific_name = str(raw.get("displayName", "") or "").strip()

        native = (
            str(raw.get("accessionVersion", "") or "").strip()
            or str(raw.get("accession", "") or "").strip()
            or accession
            or sample_accession
        )

        rows.append(
            {
                "record_id": f"pathoplexus:{native}" if native else "pathoplexus",
                "source": "pathoplexus",
                "accession": accession,
                "sample_accession": sample_accession,
                "study_accession": study_accession,
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": scientific_name,
                "tax_id": str(raw.get("ncbiVirusTaxId", "") or "").strip(),
                "country": country_field,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": str(raw.get("sampleCollectionDate", "") or "").strip(),
                "host": host,
                "isolation_source": _pathoplexus_isolation_source(raw),
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("Pathoplexus helpers ready")

Pathoplexus helpers ready


In [10]:
pathogen_by_name = pathogen_lookup()
known_insdc_accessions: set[str] = set()
for key in ("ena", "bvbrc"):
    if key not in source_frames or source_frames[key].empty:
        continue
    known_insdc_accessions.update(
        acc
        for acc in (
            normalize_accession(value) for value in source_frames[key]["accession"]
        )
        if acc
    )
print(f"Known INSDC accessions for Pathoplexus primary pick: {len(known_insdc_accessions):,}")

pathoplexus_raw_frames: list[pd.DataFrame] = []
pathoplexus_per_organism: dict[str, dict] = {}

for organism in tqdm(PATHOPLEXUS_ORGANISMS, desc="Pathoplexus organisms"):
    slug = organism["slug"]
    pathogen = organism["pathogen"]
    target = pathogen_by_name.get(pathogen)
    if target is None:
        raise KeyError(f"Pathoplexus pathogen not in VIRUS_VECTOR_TAXON_MAP: {pathogen}")

    raw = pathoplexus_sample_details(slug)
    raw_count = int(len(raw))
    if not raw.empty:
        if "isRevocation" in raw.columns:
            raw = raw.loc[~raw["isRevocation"].map(_pathoplexus_is_true)].copy()
        if "versionStatus" in raw.columns:
            raw = raw.loc[
                raw["versionStatus"].fillna("").astype(str).str.strip().eq("LATEST_VERSION")
            ].copy()
        raw["pathogen_name"] = pathogen
        raw["vector_group"] = target["vector_group"]
        raw["query_tax_id"] = str(target["tax_ids"][0])
        raw["pathoplexus_slug"] = slug
        if "accessionVersion" in raw.columns:
            raw = raw.drop_duplicates(subset=["accessionVersion"], keep="first")
        if not raw.empty:
            pathoplexus_raw_frames.append(raw)

    pathoplexus_per_organism[slug] = {
        "pathogen": pathogen,
        "vector_group": target["vector_group"],
        "tax_ids": list(target["tax_ids"]),
        "raw": raw_count,
        "latest_non_revoked": int(len(raw)) if raw_count else 0,
    }

raw_pathoplexus_df = (
    pd.concat(pathoplexus_raw_frames, ignore_index=True)
    if pathoplexus_raw_frames
    else pd.DataFrame()
)

pathoplexus_df = apply_keep_filter(
    normalize_pathoplexus_rows(
        raw_pathoplexus_df,
        known_accessions=known_insdc_accessions or None,
    )
)
pathoplexus_df = assign_record_ids(pathoplexus_df)

for slug, stats in pathoplexus_per_organism.items():
    pathogen = stats["pathogen"]
    stats["kept"] = (
        int((pathoplexus_df["pathogen_name"] == pathogen).sum())
        if len(pathoplexus_df)
        else 0
    )

source_frames["pathoplexus"] = pathoplexus_df
source_stats["pathoplexus"] = {
    "sources": ["pathoplexus"],
    "citation": PATHOPLEXUS_CITATION,
    "endpoint": f"{PATHOPLEXUS_BASE_URL}/{{slug}}/sample/details?dataFormat=tsv",
    "organisms": [o["slug"] for o in PATHOPLEXUS_ORGANISMS],
    "raw": int(sum(stats["raw"] for stats in pathoplexus_per_organism.values())),
    "latest_non_revoked": int(
        sum(stats["latest_non_revoked"] for stats in pathoplexus_per_organism.values())
    ),
    "kept": int(len(pathoplexus_df)),
    "known_insdc_accessions": len(known_insdc_accessions),
    "matched_insdc_accession": int(
        pathoplexus_df["accession"].isin(known_insdc_accessions).sum()
    )
    if len(pathoplexus_df) and known_insdc_accessions
    else 0,
    "coord_source_counts": pathoplexus_df["coord_source"]
    .value_counts(dropna=False)
    .to_dict(),
    "per_organism": pathoplexus_per_organism,
}

print(f"Pathoplexus raw rows (all versions): {source_stats['pathoplexus']['raw']:,}")
print(
    "Pathoplexus latest non-revoked: "
    f"{source_stats['pathoplexus']['latest_non_revoked']:,}"
)
print(f"Pathoplexus kept rows:             {len(pathoplexus_df):,}")
print("coord_source counts:")
print(pathoplexus_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(
    {
        slug: {
            "pathogen": stats["pathogen"],
            "raw": stats["raw"],
            "latest_non_revoked": stats["latest_non_revoked"],
            "kept": stats["kept"],
        }
        for slug, stats in pathoplexus_per_organism.items()
    }
).T

Known INSDC accessions for Pathoplexus primary pick: 122,512


Pathoplexus organisms: 100%|██████████| 6/6 [00:49<00:00,  8.27s/it]


Pathoplexus raw rows (all versions): 142,917
Pathoplexus latest non-revoked: 92,133
Pathoplexus kept rows:             87,653
coord_source counts:
coord_source
none         86879
exact_gps      774


,pathogen,raw,latest_non_revoked,kept
west-nile,West Nile virus,27555,9281,8669
dengue,Dengue virus,89520,62902,60691
chikungunya,Chikungunya virus,11504,11430,10991
zika,Zika virus,3130,3089,2259
cchf,Crimean-Congo hemorrhagic fever virus,8816,3126,3036
yellow-fever,Yellow fever virus,2392,2305,2007


## FAO EMPRES-i — BigQuery CSV

1. Query `https://api.data.apps.fao.org/api/v2/bigquery` with FAO `animal-major-disease-parameterized-query.sql`.
2. Fetch target diseases (`West Nile Fever`, `Bluetongue`, `Rift Valley fever`, `Schmallenberg`, `African horse sickness`, `Japanese Encephalitis`) for `2000-01-01` → today.
3. Drop `diagnosis_status == "Denied"`; map `global_id`, coords, country/locality/region, observation/report dates, species → host.
4. Tag `source=empres_i`, apply the place/time keep filter, register `source_frames["empres_i"]`.

Output: `empres_df`.

In [11]:
def empres_disease_csv(
    disease: str,
    *,
    start_date: str = EMPRES_START_DATE,
    end_date: str | None = None,
    diagnosis_status: str = "all",
    animal_type: str = "all",
    country: str = "all",
    timeout_s: int = DOWNLOAD_TIMEOUT_S,
) -> pd.DataFrame:
    """GET FAO EMPRES-i major-disease events as a CSV DataFrame."""
    if end_date is None:
        end_date = datetime.now(timezone.utc).date().isoformat()
    params = {
        "sql_url": EMPRES_SQL_URL,
        "disease": disease,
        "start_date": start_date,
        "end_date": end_date,
        "diagnosis_status": diagnosis_status,
        "animal_type": animal_type,
        "country": country,
    }
    url = f"{EMPRES_BIGQUERY_URL}?{urlencode(params)}"
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/csv,*/*",
        },
    )
    text = _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")
    if not text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False)


def normalize_empres_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Map EMPRES-i outbreak rows into ``VIRUS_RAW_COLUMNS`` (``source=empres_i``)."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        global_id = str(raw.get("global_id", "") or "").strip()
        country = str(raw.get("country", "") or "").strip()
        if country.lower() in {"no_value", "nan", "none", "null"}:
            country = ""
        locality = str(raw.get("locality", "") or "").strip()
        region = str(raw.get("region", "") or "").strip()
        subnational = locality or region

        lat, lon, coord_source = resolve_coords(
            lat=raw.get("lat", ""),
            lon=raw.get("lon", ""),
            coord_source="exact_gps",
        )

        collection_date = str(raw.get("observation_date", "") or "").strip()
        if not collection_date:
            collection_date = str(raw.get("report_date", "") or "").strip()
        if not collection_date:
            collection_date = str(raw.get("display_date", "") or "").strip()

        host = str(raw.get("species_overview_list", "") or "").strip()
        isolation_source = str(raw.get("animal_type_list", "") or "").strip()
        if isolation_source.lower() in {"no_value", "nan", "none", "null"}:
            isolation_source = ""

        disease_label = str(raw.get("disease", "") or "").strip()
        if disease_label.lower() in {"no_value", "nan", "none", "null"}:
            disease_label = ""

        rows.append(
            {
                "record_id": f"empres_i:{global_id}" if global_id else "empres_i",
                "source": "empres_i",
                "accession": "",
                "sample_accession": "",
                "study_accession": "",
                "pathogen_name": str(raw.get("pathogen_name", "") or "").strip(),
                "vector_group": str(raw.get("vector_group", "") or "").strip(),
                "query_tax_id": str(raw.get("query_tax_id", "") or "").strip(),
                "scientific_name": disease_label,
                "tax_id": "",
                "country": country,
                "subnational_region": subnational,
                "nuts_id": "",
                "lat": lat,
                "lon": lon,
                "coord_source": coord_source,
                "collection_date": collection_date,
                "host": host,
                "isolation_source": isolation_source,
            }
        )

    return ensure_virus_columns(pd.DataFrame(rows))


print("EMPRES-i helpers ready")

EMPRES-i helpers ready


In [12]:
pathogen_by_name = pathogen_lookup()
empres_end_date = datetime.now(timezone.utc).date().isoformat()
empres_raw_frames: list[pd.DataFrame] = []
empres_per_disease: dict[str, dict] = {}

for entry in tqdm(EMPRES_DISEASES, desc="EMPRES-i diseases"):
    disease = entry["disease"]
    pathogen = entry["pathogen"]
    target = pathogen_by_name.get(pathogen)
    if target is None:
        raise KeyError(f"EMPRES-i pathogen not in VIRUS_VECTOR_TAXON_MAP: {pathogen}")

    raw = empres_disease_csv(disease, end_date=empres_end_date)
    raw_count = int(len(raw))
    denied_count = 0
    if not raw.empty and "diagnosis_status" in raw.columns:
        denied_mask = raw["diagnosis_status"].fillna("").astype(str).str.strip().eq("Denied")
        denied_count = int(denied_mask.sum())
        raw = raw.loc[~denied_mask].copy()

    if not raw.empty:
        raw["pathogen_name"] = pathogen
        raw["vector_group"] = target["vector_group"]
        raw["query_tax_id"] = str(target["tax_ids"][0])
        raw["empres_disease"] = disease
        if "global_id" in raw.columns:
            raw = raw.drop_duplicates(subset=["global_id", "pathogen_name"], keep="first")
        empres_raw_frames.append(raw)

    empres_per_disease[disease] = {
        "pathogen": pathogen,
        "vector_group": target["vector_group"],
        "tax_ids": list(target["tax_ids"]),
        "raw": raw_count,
        "denied": denied_count,
        "after_denied_filter": int(len(raw)) if raw_count else 0,
    }

raw_empres_df = (
    pd.concat(empres_raw_frames, ignore_index=True) if empres_raw_frames else pd.DataFrame()
)

empres_df = apply_keep_filter(normalize_empres_rows(raw_empres_df))
empres_df = assign_record_ids(empres_df, native_id_column=None)

for disease, stats in empres_per_disease.items():
    pathogen = stats["pathogen"]
    stats["kept"] = (
        int((empres_df["pathogen_name"] == pathogen).sum()) if len(empres_df) else 0
    )

source_frames["empres_i"] = empres_df
source_stats["empres_i"] = {
    "sources": ["empres_i"],
    "citation": EMPRES_CITATION,
    "endpoint": EMPRES_BIGQUERY_URL,
    "sql_url": EMPRES_SQL_URL,
    "start_date": EMPRES_START_DATE,
    "end_date": empres_end_date,
    "diseases": [entry["disease"] for entry in EMPRES_DISEASES],
    "raw": int(sum(stats["raw"] for stats in empres_per_disease.values())),
    "denied": int(sum(stats["denied"] for stats in empres_per_disease.values())),
    "kept": int(len(empres_df)),
    "coord_source_counts": empres_df["coord_source"].value_counts(dropna=False).to_dict(),
    "per_disease": empres_per_disease,
}

print(f"EMPRES-i end_date: {empres_end_date}")
print(f"EMPRES-i raw rows:    {source_stats['empres_i']['raw']:,}")
print(f"EMPRES-i denied rows: {source_stats['empres_i']['denied']:,}")
print(f"EMPRES-i kept rows:   {len(empres_df):,}")
print("coord_source counts:")
print(empres_df["coord_source"].value_counts(dropna=False).to_string())
pd.DataFrame(
    {
        disease: {
            "pathogen": stats["pathogen"],
            "raw": stats["raw"],
            "denied": stats["denied"],
            "kept": stats["kept"],
        }
        for disease, stats in empres_per_disease.items()
    }
).T

EMPRES-i diseases: 100%|██████████| 6/6 [00:50<00:00,  8.46s/it]


EMPRES-i end_date: 2026-11-13
EMPRES-i raw rows:    25,889
EMPRES-i denied rows: 15
EMPRES-i kept rows:   25,874
coord_source counts:
coord_source
exact_gps    25874


,pathogen,raw,denied,kept
West Nile Fever,West Nile virus,2172,0,2172
Bluetongue,Bluetongue virus,18120,0,18120
Rift Valley fever,Rift Valley fever virus,1818,15,1803
Schmallenberg,Schmallenberg virus,3349,0,3349
African horse sickness,African horse sickness virus,240,0,240
Japanese Encephalitis,Japanese encephalitis virus,190,0,190


## ECDC — WNV weekly / monthly NUTS3 CSVs

1. Fetch `https://wnv-weekly.ecdc.europa.eu/` and `https://wnv-monthly.ecdc.europa.eu/`.
2. Decode embedded `data:text/csv;base64,...` download tables (HTML-entity unescaped).
3. Keep rows with positive human cases or equid/bird outbreaks; map country, affected area, NUTS/GAUL code, onset/report date, host summary.
4. Tag `source=ecdc_atlas` (West Nile virus only), apply keep filter, register `source_frames["ecdc_atlas"]`.

Output: `ecdc_df`.

In [13]:
_ECDC_CSV_DATA_URI_RE = re.compile(r"data:text/csv;base64,([A-Za-z0-9+/=]+)")
_ECDC_ONSET_COL = (
    "Date of onset (or proxy) first human case reported in current season (yyyy/mm/dd)"
)


def _ecdc_parse_count(value: object) -> int:
    text = str(value or "").strip()
    if not text or text.upper() in {"NA", "N/A", "NULL", "NONE", ".", "-"}:
        return 0
    try:
        return int(float(text.replace(",", "")))
    except ValueError:
        return 0


def extract_ecdc_csv_frames(html: str) -> list[pd.DataFrame]:
    """Decode all embedded ``data:text/csv;base64`` tables from an ECDC HTML page."""
    unescaped = html_module.unescape(html)
    frames: list[pd.DataFrame] = []
    for match in _ECDC_CSV_DATA_URI_RE.finditer(unescaped):
        raw = base64.b64decode(match.group(1))
        text = raw.decode("utf-8", errors="replace")
        if not text.strip():
            continue
        frames.append(pd.read_csv(io.StringIO(text), dtype=str, keep_default_na=False))
    return frames


def fetch_ecdc_html(url: str, *, timeout_s: int = DOWNLOAD_TIMEOUT_S) -> str:
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml,*/*",
        },
    )
    return _http_body(request, timeout_s=timeout_s).decode("utf-8", errors="replace")


def _ecdc_host_summary(
    *,
    human_cases: int = 0,
    equid_outbreaks: int = 0,
    bird_outbreaks: int = 0,
    animal_outbreaks: int = 0,
) -> str:
    parts: list[str] = []
    if human_cases > 0:
        parts.append(f"human:{human_cases}")
    if equid_outbreaks > 0:
        parts.append(f"equid_outbreaks:{equid_outbreaks}")
    if bird_outbreaks > 0:
        parts.append(f"bird_outbreaks:{bird_outbreaks}")
    if animal_outbreaks > 0 and equid_outbreaks == 0 and bird_outbreaks == 0:
        parts.append(f"animal_outbreaks:{animal_outbreaks}")
    return ";".join(parts)


def _normalize_ecdc_weekly_rows(
    df: pd.DataFrame,
    *,
    pathogen_name: str,
    vector_group: str,
    query_tax_id: str,
) -> pd.DataFrame:
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        human_total = _ecdc_parse_count(raw.get("Total human cases", ""))
        human_confirmed = _ecdc_parse_count(raw.get("Confirmed human cases", ""))
        human_probable = _ecdc_parse_count(raw.get("Probable human cases", ""))
        human_cases = max(human_total, human_confirmed + human_probable)
        if human_cases <= 0:
            continue

        country = str(raw.get("Country", "") or "").strip()
        nuts_id = str(raw.get("NUTS or GAUL code", "") or "").strip()
        area = str(raw.get("Affected area", "") or "").strip()
        onset = str(raw.get(_ECDC_ONSET_COL, "") or "").strip()
        if onset.upper() in {"NA", "N/A"}:
            onset = ""
        data_as_of = str(raw.get("Data as of", "") or "").strip()
        collection_date = onset or data_as_of
        native = f"weekly:{nuts_id or area}:{collection_date or data_as_of}"

        rows.append(
            {
                "record_id": f"ecdc_atlas:{native}",
                "source": "ecdc_atlas",
                "accession": "",
                "sample_accession": "",
                "study_accession": "",
                "pathogen_name": pathogen_name,
                "vector_group": vector_group,
                "query_tax_id": query_tax_id,
                "scientific_name": "West Nile virus",
                "tax_id": "",
                "country": country,
                "subnational_region": area,
                "nuts_id": nuts_id,
                "lat": "",
                "lon": "",
                "coord_source": "none",
                "collection_date": collection_date,
                "host": _ecdc_host_summary(human_cases=human_cases),
                "isolation_source": "ecdc_weekly",
            }
        )
    return ensure_virus_columns(pd.DataFrame(rows))


def _normalize_ecdc_monthly_detail_rows(
    df: pd.DataFrame,
    *,
    pathogen_name: str,
    vector_group: str,
    query_tax_id: str,
) -> pd.DataFrame:
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        human_total = _ecdc_parse_count(
            raw.get("Total number of human infections current season", "")
        )
        human_confirmed = _ecdc_parse_count(
            raw.get("Total number of confirmed human infections current season", "")
        )
        equid = _ecdc_parse_count(
            raw.get("Total number of outbreaks in equids current season", "")
        )
        birds = _ecdc_parse_count(
            raw.get("Total number of outbreaks in birds current season", "")
        )
        human_cases = max(human_total, human_confirmed)
        if human_cases <= 0 and equid <= 0 and birds <= 0:
            continue

        country = str(raw.get("Country", "") or "").strip()
        nuts_id = str(raw.get("Place of infection (NUTS3/Gaul1)", "") or "").strip()
        area = str(raw.get("Place of infection name", "") or "").strip()
        onset = str(raw.get(_ECDC_ONSET_COL, "") or "").strip()
        if onset.upper() in {"NA", "N/A"}:
            onset = ""
        data_as_of = str(raw.get("Data as of", "") or "").strip()
        collection_date = onset or data_as_of
        native = f"monthly:{nuts_id or area}:{collection_date or data_as_of}"

        rows.append(
            {
                "record_id": f"ecdc_atlas:{native}",
                "source": "ecdc_atlas",
                "accession": "",
                "sample_accession": "",
                "study_accession": "",
                "pathogen_name": pathogen_name,
                "vector_group": vector_group,
                "query_tax_id": query_tax_id,
                "scientific_name": "West Nile virus",
                "tax_id": "",
                "country": country,
                "subnational_region": area,
                "nuts_id": nuts_id,
                "lat": "",
                "lon": "",
                "coord_source": "none",
                "collection_date": collection_date,
                "host": _ecdc_host_summary(
                    human_cases=human_cases,
                    equid_outbreaks=equid,
                    bird_outbreaks=birds,
                ),
                "isolation_source": "ecdc_monthly",
            }
        )
    return ensure_virus_columns(pd.DataFrame(rows))


def _normalize_ecdc_monthly_summary_rows(
    df: pd.DataFrame,
    *,
    pathogen_name: str,
    vector_group: str,
    query_tax_id: str,
    data_as_of: str = "",
) -> pd.DataFrame:
    """Fallback for the coarser monthly summary CSV (no onset / host split)."""
    if df.empty:
        return empty_virus_frame()

    rows: list[dict[str, str]] = []
    for _, raw in df.iterrows():
        human_cases = _ecdc_parse_count(raw.get("Number of human WNV cases", ""))
        animal_outbreaks = _ecdc_parse_count(raw.get("Number of animal outbreaks", ""))
        if human_cases <= 0 and animal_outbreaks <= 0:
            continue

        country = str(raw.get("Country", "") or "").strip()
        nuts_id = str(raw.get("NUTS or GAUL code", "") or "").strip()
        area = str(raw.get("Affected area", "") or "").strip()
        collection_date = data_as_of
        native = f"monthly_summary:{nuts_id or area}:{collection_date or 'unknown'}"

        rows.append(
            {
                "record_id": f"ecdc_atlas:{native}",
                "source": "ecdc_atlas",
                "accession": "",
                "sample_accession": "",
                "study_accession": "",
                "pathogen_name": pathogen_name,
                "vector_group": vector_group,
                "query_tax_id": query_tax_id,
                "scientific_name": "West Nile virus",
                "tax_id": "",
                "country": country,
                "subnational_region": area,
                "nuts_id": nuts_id,
                "lat": "",
                "lon": "",
                "coord_source": "none",
                "collection_date": collection_date,
                "host": _ecdc_host_summary(
                    human_cases=human_cases,
                    animal_outbreaks=animal_outbreaks,
                ),
                "isolation_source": "ecdc_monthly_summary",
            }
        )
    return ensure_virus_columns(pd.DataFrame(rows))


def classify_ecdc_frames(frames: list[pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Label decoded CSVs as weekly / monthly_detail / monthly_summary."""
    out: dict[str, pd.DataFrame] = {}
    for frame in frames:
        cols = set(frame.columns)
        if {"Total human cases", "NUTS or GAUL code", "Affected area"} <= cols:
            out["weekly"] = frame
        elif {
            "Place of infection (NUTS3/Gaul1)",
            "Total number of outbreaks in equids current season",
        } <= cols:
            out["monthly_detail"] = frame
        elif {"Number of human WNV cases", "Number of animal outbreaks"} <= cols:
            out["monthly_summary"] = frame
    return out


print("ECDC helpers ready")

ECDC helpers ready


In [14]:
pathogen_by_name = pathogen_lookup()
ecdc_target = pathogen_by_name[ECDC_PATHOGEN]
ecdc_query_tax_id = str(ecdc_target["tax_ids"][0])
ecdc_vector_group = ecdc_target["vector_group"]

ecdc_weekly_html = fetch_ecdc_html(ECDC_WEEKLY_URL)
ecdc_monthly_html = fetch_ecdc_html(ECDC_MONTHLY_URL)

ecdc_weekly_frames = extract_ecdc_csv_frames(ecdc_weekly_html)
ecdc_monthly_frames = extract_ecdc_csv_frames(ecdc_monthly_html)
ecdc_tables = {
    **{f"weekly_{i}": frame for i, frame in enumerate(ecdc_weekly_frames)},
    **{f"monthly_{i}": frame for i, frame in enumerate(ecdc_monthly_frames)},
}
classified = classify_ecdc_frames([*ecdc_weekly_frames, *ecdc_monthly_frames])

print(
    "Decoded CSV tables:",
    {name: list(frame.columns)[:4] + [f"...({len(frame)} rows)"] for name, frame in classified.items()},
)

ecdc_parts: list[pd.DataFrame] = []
ecdc_feed_stats: dict[str, dict] = {}

if "weekly" in classified:
    weekly_raw = classified["weekly"]
    weekly_norm = apply_keep_filter(
        _normalize_ecdc_weekly_rows(
            weekly_raw,
            pathogen_name=ECDC_PATHOGEN,
            vector_group=ecdc_vector_group,
            query_tax_id=ecdc_query_tax_id,
        )
    )
    ecdc_parts.append(weekly_norm)
    ecdc_feed_stats["weekly"] = {
        "raw_rows": int(len(weekly_raw)),
        "kept": int(len(weekly_norm)),
        "endpoint": ECDC_WEEKLY_URL,
    }

if "monthly_detail" in classified:
    monthly_raw = classified["monthly_detail"]
    monthly_norm = apply_keep_filter(
        _normalize_ecdc_monthly_detail_rows(
            monthly_raw,
            pathogen_name=ECDC_PATHOGEN,
            vector_group=ecdc_vector_group,
            query_tax_id=ecdc_query_tax_id,
        )
    )
    ecdc_parts.append(monthly_norm)
    ecdc_feed_stats["monthly_detail"] = {
        "raw_rows": int(len(monthly_raw)),
        "kept": int(len(monthly_norm)),
        "endpoint": ECDC_MONTHLY_URL,
    }
elif "monthly_summary" in classified:
    # Coarser fallback only when the detailed monthly table is unavailable.
    summary_raw = classified["monthly_summary"]
    summary_norm = apply_keep_filter(
        _normalize_ecdc_monthly_summary_rows(
            summary_raw,
            pathogen_name=ECDC_PATHOGEN,
            vector_group=ecdc_vector_group,
            query_tax_id=ecdc_query_tax_id,
        )
    )
    ecdc_parts.append(summary_norm)
    ecdc_feed_stats["monthly_summary"] = {
        "raw_rows": int(len(summary_raw)),
        "kept": int(len(summary_norm)),
        "endpoint": ECDC_MONTHLY_URL,
    }

ecdc_df = (
    ensure_virus_columns(pd.concat(ecdc_parts, ignore_index=True))
    if ecdc_parts
    else empty_virus_frame()
)
ecdc_df = assign_record_ids(ecdc_df)

source_frames["ecdc_atlas"] = ecdc_df
source_stats["ecdc_atlas"] = {
    "sources": ["ecdc_atlas"],
    "citation": ECDC_CITATION,
    "endpoints": [ECDC_WEEKLY_URL, ECDC_MONTHLY_URL],
    "pathogen": ECDC_PATHOGEN,
    "decoded_tables": sorted(classified),
    "raw": int(sum(stats["raw_rows"] for stats in ecdc_feed_stats.values())),
    "kept": int(len(ecdc_df)),
    "coord_source_counts": ecdc_df["coord_source"].value_counts(dropna=False).to_dict(),
    "feeds": ecdc_feed_stats,
    "isolation_source_counts": ecdc_df["isolation_source"]
    .value_counts(dropna=False)
    .to_dict(),
}

print(f"ECDC kept rows: {len(ecdc_df):,}")
print("Rows by feed (isolation_source):")
print(ecdc_df["isolation_source"].value_counts(dropna=False).to_string())
print("Sample host summaries:")
print(ecdc_df["host"].value_counts().head(10).to_string())
ecdc_df.head(5)

Decoded CSV tables: {'weekly': ['Country', 'NUTS or GAUL code', 'Affected area', 'First reported this week*', '...(235 rows)'], 'monthly_summary': ['Region', 'Country', 'Affected area', 'NUTS or GAUL code', '...(313 rows)'], 'monthly_detail': ['Country', 'Place of infection (NUTS3/Gaul1)', 'Place of infection name', 'Date of onset (or proxy) first human case reported in current season (yyyy/mm/dd)', '...(313 rows)']}
ECDC kept rows: 548
Rows by feed (isolation_source):
isolation_source
ecdc_monthly    313
ecdc_weekly     235
Sample host summaries:
host
human:1              128
human:2               46
human:3               30
human:5               26
equid_outbreaks:1     25
human:4               21
bird_outbreaks:1      14
human:6               12
human:7                8
human:8                7


,record_id,source,accession,sample_accession,study_accession,pathogen_name,vector_group,query_tax_id,scientific_name,tax_id,country,subnational_region,nuts_id,lat,lon,coord_source,collection_date,host,isolation_source
0,ecdc_atlas:weekly:ITI43:2026-07-17,ecdc_atlas,,,,West Nile virus,mosquito,3048448,West Nile virus,,Italy,Roma,ITI43,,,none,2026-07-17,human:72,ecdc_weekly
1,ecdc_atlas:weekly:ITH36:2026-07-05,ecdc_atlas,,,,West Nile virus,mosquito,3048448,West Nile virus,,Italy,Padova,ITH36,,,none,2026-07-05,human:63,ecdc_weekly
2,ecdc_atlas:weekly:ITC4C:2026-07-01,ecdc_atlas,,,,West Nile virus,mosquito,3048448,West Nile virus,,Italy,Milano,ITC4C,,,none,2026-07-01,human:60,ecdc_weekly
3,ecdc_atlas:weekly:ITI44:2026-07-01,ecdc_atlas,,,,West Nile virus,mosquito,3048448,West Nile virus,,Italy,Latina,ITI44,,,none,2026-07-01,human:36,ecdc_weekly
4,ecdc_atlas:weekly:ITC11:2026-07-15,ecdc_atlas,,,,West Nile virus,mosquito,3048448,West Nile virus,,Italy,Torino,ITC11,,,none,2026-07-15,human:36,ecdc_weekly


## Merge, deduplicate, and write unified extract

1. Concatenate all source frames (`ena`, `bvbrc`, `pathoplexus`, `empres_i`, `ecdc_atlas`).
2. Run `dedupe_virus_detections` (INSDC group merge by `accession`, else `sample_accession` + pathogen; outbreak dedupe for `empres_i` and `ecdc_atlas`).
3. Assign stable `record_id` (`{source}:{native_id}`) and write `data/01_raw/virus/virus_detections.tsv` + `manifest.json` (`fetched_at`, SHA-256, per-source and per-pathogen counts, citations, `coord_source_counts`, targets).

In [15]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


SOURCE_ORDER: tuple[str, ...] = ("ena", "bvbrc", "pathoplexus", "empres_i", "ecdc_atlas")
SOURCE_CITATIONS: dict[str, str] = {
    "ena_sequence": ENA_CITATION,
    "ena_read_run": ENA_CITATION,
    "bvbrc": BVBRC_CITATION,
    "pathoplexus": PATHOPLEXUS_CITATION,
    "empres_i": EMPRES_CITATION,
    "ecdc_atlas": ECDC_CITATION,
}

raw_parts = [
    assign_record_ids(ensure_virus_columns(source_frames[key]))
    for key in SOURCE_ORDER
    if key in source_frames and not source_frames[key].empty
]
combined_pre_dedupe_df = (
    ensure_virus_columns(pd.concat(raw_parts, ignore_index=True))
    if raw_parts
    else empty_virus_frame()
)
pre_dedupe_by_source = (
    combined_pre_dedupe_df["source"].value_counts(dropna=False).to_dict()
    if len(combined_pre_dedupe_df)
    else {}
)

df = assign_record_ids(dedupe_virus_detections(combined_pre_dedupe_df))
post_dedupe_by_source = (
    df["source"].value_counts(dropna=False).to_dict() if len(df) else {}
)

df.to_csv(TSV_PATH, sep="\t", index=False)

insdc_pre = int(combined_pre_dedupe_df["source"].isin(INSDC_SOURCES).sum())
insdc_post = int(df["source"].isin(INSDC_SOURCES).sum())
outbreak_pre = int(combined_pre_dedupe_df["source"].isin(OUTBREAK_SOURCES).sum())
outbreak_post = int(df["source"].isin(OUTBREAK_SOURCES).sum())

manifest = {
    "fetched_at": datetime.now(timezone.utc).isoformat(),
    "tsv_name": TSV_NAME,
    "tsv_bytes": TSV_PATH.stat().st_size,
    "sha256": sha256_file(TSV_PATH),
    "pre_dedupe_row_count": int(len(combined_pre_dedupe_df)),
    "row_count": int(len(df)),
    "columns": list(VIRUS_RAW_COLUMNS),
    "filter": (
        "keep rows with non-empty country, nuts_id, (lat,lon), or collection_date; "
        "ENA sample backfill applied before keep filter; "
        "INSDC rows merged by normalized accession else sample_accession (+pathogen when multi-virus); "
        "outbreak rows deduplicated by native event_id or pathogen+loc+season"
    ),
    "citations": SOURCE_CITATIONS,
    "dedupe_summary": {
        "pre_dedupe_total": int(len(combined_pre_dedupe_df)),
        "post_dedupe_total": int(len(df)),
        "collapsed_rows": int(len(combined_pre_dedupe_df) - len(df)),
        "insdc_pre_dedupe": insdc_pre,
        "insdc_post_dedupe": insdc_post,
        "insdc_collapsed": int(insdc_pre - insdc_post),
        "outbreak_pre_dedupe": outbreak_pre,
        "outbreak_post_dedupe": outbreak_post,
        "outbreak_collapsed": int(outbreak_pre - outbreak_post),
        "pre_dedupe_by_source": {k: int(v) for k, v in pre_dedupe_by_source.items()},
        "post_dedupe_by_source": {k: int(v) for k, v in post_dedupe_by_source.items()},
    },
    "per_source": {k: int(v) for k, v in post_dedupe_by_source.items()},
    "sources": source_stats,
    "coord_source_counts": {
        k: int(v) for k, v in df["coord_source"].value_counts(dropna=False).to_dict().items()
    },
    "per_pathogen": {
        pathogen: int((df["pathogen_name"] == pathogen).sum())
        for pathogen in sorted(df["pathogen_name"].unique())
    },
    "per_vector_group": {
        vg: int((df["vector_group"] == vg).sum())
        for vg in sorted(df["vector_group"].unique())
    },
    "targets": [
        {
            "pathogen": t["pathogen"],
            "tax_ids": list(t["tax_ids"]),
            "legacy_tax_ids": list(t.get("legacy_tax_ids", ())),
            "vector_group": t["vector_group"],
        }
        for t in TARGETS
    ],
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print(f"Pre-dedupe rows:  {len(combined_pre_dedupe_df):,}")
print(f"Post-dedupe rows: {len(df):,} (collapsed {len(combined_pre_dedupe_df) - len(df):,})")
print(f"  INSDC:    {insdc_pre:,} -> {insdc_post:,} (collapsed {insdc_pre - insdc_post:,})")
print(f"  Outbreak: {outbreak_pre:,} -> {outbreak_post:,} (collapsed {outbreak_pre - outbreak_post:,})")
print(f"Wrote {TSV_PATH} ({manifest['tsv_bytes']:,} bytes)")
print(f"Wrote {MANIFEST_PATH}")
print("\nRows by source (pre -> post dedupe):")
all_sources = list(
    dict.fromkeys([*INSDC_SOURCES, *OUTBREAK_SOURCES, *pre_dedupe_by_source, *post_dedupe_by_source])
)
print(
    pd.DataFrame(
        {
            "pre_dedupe": {s: int(pre_dedupe_by_source.get(s, 0)) for s in all_sources},
            "post_dedupe": {s: int(post_dedupe_by_source.get(s, 0)) for s in all_sources},
        }
    ).to_string()
)
print("\ncoord_source counts (post-dedupe):")
print(df["coord_source"].value_counts(dropna=False).to_string())
manifest["sha256"], manifest["row_count"]

Pre-dedupe rows:  374,931
Post-dedupe rows: 168,307 (collapsed 206,624)
  INSDC:    348,509 -> 142,120 (collapsed 206,389)
  Outbreak: 26,422 -> 26,187 (collapsed 235)
Wrote /workspaces/vector-guard/data/01_raw/virus/virus_detections.tsv (28,564,466 bytes)
Wrote /workspaces/vector-guard/data/01_raw/virus/manifest.json

Rows by source (pre -> post dedupe):
              pre_dedupe  post_dedupe
ena_sequence      118407       118404
ena_read_run       17111        13231
bvbrc             125338         4108
pathoplexus        87653         6377
empres_i           25874        25874
ecdc_atlas           548          313

coord_source counts (post-dedupe):
coord_source
none               132557
exact_gps           34187
sample_backfill      1563


('12b1d3ffdb864380c9ecfad1d270e4513c60cc99e2c6e6029e0cb05c89a81ce3', 168307)

## Smoke check

Reload via `load_virus_detections` and confirm the unified schema, `record_id` uniqueness, `source` mix, and coordinate coverage.

In [16]:
print("Files in raw dir:")
for rel in sorted(
    p.relative_to(RAW_DIR).as_posix() for p in RAW_DIR.rglob("*") if p.is_file()
):
    print(f"  - {rel}")

core_path = resolve_virus_core_path(RAW_DIR)
loaded = load_virus_detections(RAW_DIR)
print(f"\nCore file: {core_path.name}")
print(f"Rows: {len(loaded):,}")
print(f"Columns ({len(loaded.columns)}): {list(loaded.columns)}")
missing_cols = [c for c in VIRUS_RAW_COLUMNS if c not in loaded.columns]
print(f"Missing canonical columns: {missing_cols or 'none'}")
assert list(loaded.columns) == list(VIRUS_RAW_COLUMNS), "Column order mismatch"
assert (loaded["record_id"].str.strip() != "").all(), "Blank record_id found"
has_coords = (loaded["lat"].str.strip() != "") & (loaded["lon"].str.strip() != "")
has_nuts = loaded["nuts_id"].str.strip() != ""
print(
    f"Rows with decimal (lat, lon): {int(has_coords.sum()):,} "
    f"({100.0 * has_coords.mean():.2f}%)"
)
print(
    f"Rows with nuts_id:            {int(has_nuts.sum()):,} "
    f"({100.0 * has_nuts.mean():.2f}%)"
)
print("\nRows by source:")
print(loaded["source"].value_counts().to_string())
print("\nRows by coord_source:")
print(loaded["coord_source"].value_counts().to_string())
print("\nCoord coverage by source:")
print(
    loaded.assign(has_coords=has_coords, has_nuts=has_nuts)
    .groupby("source", sort=False)
    .agg(
        rows=("record_id", "size"),
        with_coords=("has_coords", "sum"),
        with_nuts=("has_nuts", "sum"),
    )
    .to_string()
)
print("\nRows by pathogen_name:")
print(loaded["pathogen_name"].value_counts().to_string())
print("\nRows by vector_group:")
print(loaded["vector_group"].value_counts().to_string())
loaded.head(8).T

Files in raw dir:
  - .gitkeep
  - manifest.json
  - virus_detections.tsv

Core file: virus_detections.tsv
Rows: 168,307
Columns (19): ['record_id', 'source', 'accession', 'sample_accession', 'study_accession', 'pathogen_name', 'vector_group', 'query_tax_id', 'scientific_name', 'tax_id', 'country', 'subnational_region', 'nuts_id', 'lat', 'lon', 'coord_source', 'collection_date', 'host', 'isolation_source']
Missing canonical columns: none
Rows with decimal (lat, lon): 35,750 (21.24%)
Rows with nuts_id:            313 (0.19%)

Rows by source:
source
ena_sequence    118404
empres_i         25874
ena_read_run     13231
pathoplexus       6377
bvbrc             4108
ecdc_atlas         313

Rows by coord_source:
coord_source
none               132557
exact_gps           34187
sample_backfill      1563

Coord coverage by source:
                rows  with_coords  with_nuts
source                                      
ena_sequence  118404         5482          0
ena_read_run   13231         425

,0,1,2,3,4,5,6,7
record_id,ena_sequence:MH170264,ena_sequence:MH819470,ena_sequence:MW803141,ena_sequence:MW835355,ena_sequence:MW835358,ena_sequence:MW862090,ena_sequence:MW862097,ena_sequence:MW862099
source,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence,ena_sequence
accession,MH170264,MH819470,MW803141,MW835355,MW835358,MW862090,MW862097,MW862099
sample_accession,,,,,,,,
study_accession,,PRJNA480662,,,,,,
pathogen_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
vector_group,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito,mosquito
query_tax_id,3048448,3048448,3048448,3048448,3048448,3048448,3048448,3048448
scientific_name,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus,West Nile virus
tax_id,11082,11082,11082,11082,11082,11082,11082,11082
